In [256]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

from sklearn.metrics import confusion_matrix

from scipy.special import softmax
from sklearn.metrics import log_loss
from scipy.optimize import differential_evolution

In [257]:
ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
OOF_DIR = ROOT_DIR / "oof"

SUBMISSION_DIR = ROOT_DIR / "submission"
SUBMISSION_DIR.mkdir(exist_ok=True)

In [258]:
conv_logits_oof = np.load(OOF_DIR / "convnext_oof_logits.npy")
swin_logits_oof = np.load(OOF_DIR / "swin_oof_logits.npy")
siglip_logits_oof = np.load(OOF_DIR / "siglip_oof_logits.npy")

In [259]:
conv_logits_test = np.load(OOF_DIR / "convnext_test_logits.npy")
swin_logits_test = np.load(OOF_DIR / "swin_test_logits.npy")
siglip_logits_test = np.load(OOF_DIR / "siglip_test_logits.npy")

In [260]:
p_conv_oof = softmax(conv_logits_oof, axis=1)
p_swin_oof = softmax(swin_logits_oof, axis=1)
p_siglip_oof = softmax(siglip_logits_oof, axis=1)

In [261]:
p_conv_test = softmax(conv_logits_test, axis=1)
p_swin_test = softmax(swin_logits_test, axis=1)
p_siglip_test = softmax(siglip_logits_test, axis=1)

In [262]:
train_folds = pd.read_csv("train_folds.csv")

y_true = train_folds["label"].to_numpy()

In [263]:
print("Train samples :", len(y_true))
print()

print("ConvNeXt OOF :", p_conv_oof.shape)
print("Swin OOF     :", p_swin_oof.shape)
print("SigLIP OOF   :", p_siglip_oof.shape)

print()

print("ConvNeXt Test :", p_conv_test.shape)
print("Swin Test     :", p_swin_test.shape)
print("SigLIP Test   :", p_siglip_test.shape)

Train samples : 24853

ConvNeXt OOF : (24853, 3)
Swin OOF     : (24853, 3)
SigLIP OOF   : (24853, 3)

ConvNeXt Test : (1458, 3)
Swin Test     : (1458, 3)
SigLIP Test   : (1458, 3)


In [264]:
pred_conv = p_conv_oof.argmax(axis=1)
pred_swin = p_swin_oof.argmax(axis=1)
pred_siglip = p_siglip_oof.argmax(axis=1)

In [265]:
best = fine_df.iloc[0]


FINAL_WEIGHTS = np.array([
    best["ConvNeXt"],
    best["Swin"],
    best["SigLIP"]
])


print(FINAL_WEIGHTS)

[0.45 0.43 0.12]


In [266]:
acc_conv = accuracy_score(y_true, pred_conv)
acc_swin = accuracy_score(y_true, pred_swin)
acc_siglip = accuracy_score(y_true, pred_siglip)

print(f"ConvNeXt Accuracy : {acc_conv:.6f}")
print(f"Swin Accuracy     : {acc_swin:.6f}")
print(f"SigLIP Accuracy   : {acc_siglip:.6f}")

ConvNeXt Accuracy : 0.973645
Swin Accuracy     : 0.973404
SigLIP Accuracy   : 0.957993


In [267]:
FINAL_WEIGHTS = np.array([
    best["ConvNeXt"],
    best["Swin"],
    best["SigLIP"]
])

p_weighted = (
    FINAL_WEIGHTS[0] * p_conv_oof +
    FINAL_WEIGHTS[1] * p_swin_oof +
    FINAL_WEIGHTS[2] * p_siglip_oof
)

pred_weighted = p_weighted.argmax(axis=1)

acc_weighted = accuracy_score(y_true, pred_weighted)

print(f"\nWeighted Ensemble Accuracy : {acc_weighted:.6f}")


Weighted Ensemble Accuracy : 0.981089


In [268]:
baseline_df = pd.DataFrame({
    "Method": [
        "ConvNeXt",
        "Swin",
        "SigLIP",
        "Weighted Ensemble"
    ],
    "Accuracy": [
        acc_conv,
        acc_swin,
        acc_siglip,
        acc_weighted
    ]
})

baseline_df = baseline_df.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

baseline_df["Accuracy"] = baseline_df["Accuracy"].map(lambda x: f"{x:.6%}")

baseline_df

,Method,Accuracy
0,Weighted Ensemble,98.108880%
1,ConvNeXt,97.364503%
2,Swin,97.340361%
3,SigLIP,95.799300%


In [269]:
agreement_df = pd.DataFrame({
    "ConvNeXt": pred_conv,
    "Swin": pred_swin,
    "SigLIP": pred_siglip
})

agreement_df["unique_predictions"] = agreement_df.nunique(axis=1)

agreement_summary = (
    agreement_df["unique_predictions"]
    .value_counts()
    .sort_index()
    .rename(index={
        1: "3 Models Agree",
        2: "2 Models Agree",
        3: "All Different"
    })
    .to_frame("Samples")
)

agreement_summary["Percentage"] = (
    agreement_summary["Samples"] / len(y_true) * 100
).round(2)

agreement_summary


,Samples,Percentage
unique_predictions,,
3 Models Agree,23388,94.11
2 Models Agree,1441,5.80
All Different,24,0.10


In [270]:
def evaluate_weights(weights):
    probs = (
        weights[0] * p_conv_oof +
        weights[1] * p_swin_oof +
        weights[2] * p_siglip_oof
    )

    pred = probs.argmax(axis=1)

    return accuracy_score(y_true, pred)

In [271]:
FINAL_WEIGHTS = np.array([
    best["ConvNeXt"],
    best["Swin"],
    best["SigLIP"]
])

In [272]:
results = []

for w1 in np.arange(0, 1.01, 0.05):

    for w2 in np.arange(0, 1.01 - w1, 0.05):

        w3 = 1 - w1 - w2

        if w3 < 0:
            continue

        weights = np.array([w1, w2, w3])

        acc = evaluate_weights(weights)

        results.append({
            "ConvNeXt": w1,
            "Swin": w2,
            "SigLIP": w3,
            "Accuracy": acc
        })

In [273]:
grid_df = pd.DataFrame(results)

grid_df = grid_df.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

grid_df.head(20)

,ConvNeXt,Swin,SigLIP,Accuracy
0,0.45,0.35,0.20,0.980968
1,0.45,0.40,0.15,0.980888
2,0.50,0.45,0.05,0.980888
3,0.45,0.45,0.10,0.980847
4,0.50,0.40,0.10,0.980646
5,0.40,0.35,0.25,0.980566
6,0.40,0.45,0.15,0.980445
7,0.45,0.30,0.25,0.980405
8,0.45,0.50,0.05,0.980365
9,0.50,0.30,0.20,0.980284


In [274]:
best = grid_df.iloc[0]

print(best)

ConvNeXt    0.450000
Swin        0.350000
SigLIP      0.200000
Accuracy    0.980968
Name: 0, dtype: float64


In [275]:
FINAL_WEIGHTS = np.array([
    best["ConvNeXt"],
    best["Swin"],
    best["SigLIP"]
])

In [276]:
fine_results = []

for w1 in np.arange(0.35, 0.56, 0.01):

    for w2 in np.arange(0.30, 0.51, 0.01):

        w3 = 1 - w1 - w2

        if w3 < 0 or w3 > 1:
            continue

        weights = np.array([w1, w2, w3])

        acc = evaluate_weights(weights)

        fine_results.append({
            "ConvNeXt": round(w1, 2),
            "Swin": round(w2, 2),
            "SigLIP": round(w3, 2),
            "Accuracy": acc
        })

In [277]:
fine_df = (
    pd.DataFrame(fine_results)
      .sort_values("Accuracy", ascending=False)
      .reset_index(drop=True)
)

fine_df.head(20)

,ConvNeXt,Swin,SigLIP,Accuracy
0,0.45,0.43,0.12,0.981089
1,0.44,0.45,0.11,0.981049
2,0.44,0.35,0.21,0.981008
3,0.44,0.44,0.12,0.981008
4,0.45,0.46,0.09,0.980968
5,0.43,0.43,0.14,0.980968
6,0.45,0.35,0.20,0.980968
7,0.44,0.43,0.13,0.980968
8,0.47,0.46,0.07,0.980968
9,0.47,0.42,0.11,0.980968


In [278]:
best_fine = fine_df.iloc[0]

print(best_fine)

ConvNeXt    0.450000
Swin        0.430000
SigLIP      0.120000
Accuracy    0.981089
Name: 0, dtype: float64


In [279]:
def adaptive_ensemble(p_conv, p_swin, p_siglip,
                      base_weights=np.array([0.45, 0.43, 0.12])):

    probs = np.stack([
        p_conv,
        p_swin,
        p_siglip
    ], axis=1)

    confidence = probs.max(axis=2)

    adaptive_weights = confidence * base_weights

    adaptive_weights /= adaptive_weights.sum(
        axis=1,
        keepdims=True
    )

    ensemble = (
        adaptive_weights[:,0,None] * p_conv +
        adaptive_weights[:,1,None] * p_swin +
        adaptive_weights[:,2,None] * p_siglip
    )

    return ensemble

In [280]:
p_adaptive = adaptive_ensemble(
    p_conv_oof,
    p_swin_oof,
    p_siglip_oof
)

pred_adaptive = p_adaptive.argmax(axis=1)

acc_adaptive = accuracy_score(
    y_true,
    pred_adaptive
)

print(acc_adaptive)

0.9809680923832133


In [281]:
comparison = pd.DataFrame({
    "Method":[
        "Baseline",
        "Fine Grid",
        "Adaptive"
    ],
    "Accuracy":[
        baseline_acc,
        best_fine["Accuracy"],
        acc_adaptive
    ]
})

comparison.sort_values(
    "Accuracy",
    ascending=False
)

,Method,Accuracy
0,Baseline,0.981089
1,Fine Grid,0.981089
2,Adaptive,0.980968


In [282]:
X_meta = np.concatenate([
    p_conv_oof,
    p_swin_oof,
    p_siglip_oof
], axis=1)

X_test = np.concatenate([
    p_conv_test,
    p_swin_test,
    p_siglip_test
], axis=1)

print(X_meta.shape)
print(X_test.shape)

(24853, 9)
(1458, 9)


In [283]:
oof_pred = np.zeros((len(train_folds), 3))

test_pred = np.zeros((len(X_test), 3))

fold_scores = []

In [284]:
for fold in sorted(train_folds.fold.unique()):

    print("="*40)
    print("Fold", fold)

    train_idx = train_folds.fold != fold
    valid_idx = train_folds.fold == fold

    X_train = X_meta[train_idx]
    y_train = y_true[train_idx]

    X_valid = X_meta[valid_idx]
    y_valid = y_true[valid_idx]

    meta = LogisticRegression(
        max_iter=5000,
        random_state=42
    )

    meta.fit(X_train, y_train)

    valid_prob = meta.predict_proba(X_valid)

    oof_pred[valid_idx] = valid_prob

    pred = valid_prob.argmax(axis=1)

    score = accuracy_score(y_valid, pred)

    fold_scores.append(score)

    print("Accuracy:", score)

    test_pred += meta.predict_proba(X_test) / train_folds.fold.nunique()

Fold 0
Accuracy: 0.9831019915509958
Fold 1
Accuracy: 0.9820961577147456
Fold 2
Accuracy: 0.9820961577147456
Fold 3
Accuracy: 0.9806841046277666
Fold 4
Accuracy: 0.9766599597585514


In [285]:
meta_pred = oof_pred.argmax(axis=1)

meta_acc = accuracy_score(
    y_true,
    meta_pred
)

print()
print("="*50)
print("Meta OOF Accuracy :", meta_acc)
print("Fold Mean         :", np.mean(fold_scores))


Meta OOF Accuracy : 0.9809278557920573
Fold Mean         : 0.9809276742733608


In [286]:
comparison = pd.DataFrame({

    "Method":[
        "Baseline",
        "Fine Grid",
        "Meta Stacking"
    ],

    "Accuracy":[
        baseline_acc,
        best_fine["Accuracy"],
        meta_acc
    ]

})

comparison.sort_values(
    "Accuracy",
    ascending=False
)

,Method,Accuracy
0,Baseline,0.981089
1,Fine Grid,0.981089
2,Meta Stacking,0.980928


In [287]:
def objective(w):
    # pastikan bobot positif dan jumlah = 1
    w = np.maximum(w, 0)
    w = w / w.sum()

    probs = (
        w[0] * p_conv_oof +
        w[1] * p_swin_oof +
        w[2] * p_siglip_oof
    )

    pred = probs.argmax(axis=1)

    return -accuracy_score(y_true, pred)


result = differential_evolution(
    objective,
    bounds=[(0,1), (0,1), (0,1)],
    seed=42,
    maxiter=100,
    polish=True
)

best_weights = np.maximum(result.x, 0)
best_weights /= best_weights.sum()

print("Best weights:")
print(np.round(best_weights, 6))
print("Sum =", best_weights.sum())

p_final = (
    best_weights[0] * p_conv_oof +
    best_weights[1] * p_swin_oof +
    best_weights[2] * p_siglip_oof
)

pred = p_final.argmax(1)

acc = accuracy_score(y_true, pred)

print("Accuracy:", acc)

Best weights:
[0.497297 0.466007 0.036696]
Sum = 0.9999999999999999
Accuracy: 0.9810485655655253


In [288]:
best_weights = result.x
best_weights /= best_weights.sum()

print(best_weights)

p_final = (
    best_weights[0] * p_conv_oof +
    best_weights[1] * p_swin_oof +
    best_weights[2] * p_siglip_oof
)

[0.49729654 0.46600696 0.0366965 ]


In [289]:
pred = p_final.argmax(axis=1)

acc = accuracy_score(y_true, pred)

print(acc)

0.9810485655655253


In [290]:
comparison = pd.DataFrame({

    "Method":[
        "Baseline",
        "Fine Grid",
        "SciPy Optimize"
    ],

    "Accuracy":[
        baseline_acc,
        best_fine["Accuracy"],
        acc
    ]

})

comparison.sort_values(
    "Accuracy",
    ascending=False
)

,Method,Accuracy
0,Baseline,0.981089
1,Fine Grid,0.981089
2,SciPy Optimize,0.981049


In [291]:
X_meta = np.concatenate([
    p_conv_oof,
    p_swin_oof,
    p_siglip_oof
], axis=1)

X_test = np.concatenate([
    p_conv_test,
    p_swin_test,
    p_siglip_test
], axis=1)

print(X_meta.shape)
print(X_test.shape)

(24853, 9)
(1458, 9)


In [292]:
oof_prob = np.zeros((len(train_folds), 3))

test_prob = np.zeros((len(X_test), 3))

fold_scores = []

In [293]:
for fold in sorted(train_folds.fold.unique()):

    print("="*50)
    print("Fold", fold)

    train_idx = train_folds.fold != fold
    valid_idx = train_folds.fold == fold

    X_train = X_meta[train_idx]
    y_train = y_true[train_idx]

    X_valid = X_meta[valid_idx]
    y_valid = y_true[valid_idx]

    model = CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        verbose=False,
        random_seed=42
    )

    model.fit(
        X_train,
        y_train
    )

    valid_prob = model.predict_proba(X_valid)

    oof_prob[valid_idx] = valid_prob

    pred = valid_prob.argmax(axis=1)

    score = accuracy_score(
        y_valid,
        pred
    )

    fold_scores.append(score)

    print("Accuracy:", score)

    test_prob += model.predict_proba(X_test) / train_folds.fold.nunique()

Fold 0
Accuracy: 0.9839066586199959
Fold 1
Accuracy: 0.9806879903439951
Fold 2
Accuracy: 0.9818949909474954
Fold 3
Accuracy: 0.9812877263581489
Fold 4
Accuracy: 0.9768611670020121


In [294]:
pred = oof_prob.argmax(axis=1)

cat_acc = accuracy_score(
    y_true,
    pred
)

print("="*60)
print("CatBoost Accuracy :", cat_acc)
print("Fold Mean         :", np.mean(fold_scores))

CatBoost Accuracy : 0.9809278557920573
Fold Mean         : 0.9809277066543294


In [295]:
comparison = pd.DataFrame({

    "Method":[
        "Baseline",
        "Fine Grid",
        "Logistic",
        "CatBoost"
    ],

    "Accuracy":[
        baseline_acc,
        best_fine["Accuracy"],
        meta_acc,
        cat_acc
    ]

})

comparison = comparison.sort_values(
    "Accuracy",
    ascending=False
)

comparison

,Method,Accuracy
0,Baseline,0.981089
1,Fine Grid,0.981089
2,Logistic,0.980928
3,CatBoost,0.980928


In [296]:
pred_conv = p_conv_oof.argmax(1)
pred_swin = p_swin_oof.argmax(1)
pred_sig  = p_siglip_oof.argmax(1)

agree_type = []

for a,b,c in zip(pred_conv,pred_swin,pred_sig):

    if a==b==c:
        agree_type.append(3)

    elif len({a,b,c})==2:
        agree_type.append(2)

    else:
        agree_type.append(1)

agree_type = np.array(agree_type)

In [ ]:
FINAL_WEIGHTS = np.array([
    best["ConvNeXt"],
    best["Swin"],
    best["SigLIP"]
])

ensemble_prob = (
      weights[0]*p_conv_oof
    + weights[1]*p_swin_oof
    + weights[2]*p_siglip_oof
)

final_pred = ensemble_prob.argmax(1)

In [298]:
conf_conv = p_conv_oof.max(1)
conf_swin = p_swin_oof.max(1)
conf_sig  = p_siglip_oof.max(1)

In [299]:
margin = 0.15

for i in np.where(agree_type==2)[0]:

    preds = np.array([
        pred_conv[i],
        pred_swin[i],
        pred_sig[i]
    ])

    confs = np.array([
        conf_conv[i],
        conf_swin[i],
        conf_sig[i]
    ])

    values,counts = np.unique(preds,return_counts=True)

    majority = values[np.argmax(counts)]

    minority = values[np.argmin(counts)]

    maj_conf = confs[preds==majority].mean()

    min_conf = confs[preds==minority][0]

    if min_conf > maj_conf + margin:

        final_pred[i]=minority

In [300]:
for i in np.where(agree_type==1)[0]:

    confs = np.array([
        conf_conv[i],
        conf_swin[i],
        conf_sig[i]
    ])

    preds = np.array([
        pred_conv[i],
        pred_swin[i],
        pred_sig[i]
    ])

    final_pred[i]=preds[np.argmax(confs)]

In [301]:
acc_rule = accuracy_score(
    y_true,
    final_pred
)

print(acc_rule)

0.9771858528145495


In [302]:
baseline_correct = (ensemble_prob.argmax(1)==y_true)
rule_correct = (final_pred==y_true)

improved = np.sum(~baseline_correct & rule_correct)

worsened = np.sum(baseline_correct & ~rule_correct)

print("Improved :", improved)
print("Worsened :", worsened)

Improved : 32
Worsened : 76


In [303]:
# Fine Grid weights
weights = np.array([0.45, 0.43, 0.12])

ensemble_prob = (
    weights[0] * p_conv_oof +
    weights[1] * p_swin_oof +
    weights[2] * p_siglip_oof
)

ensemble_pred = ensemble_prob.argmax(axis=1)

error_df = pd.DataFrame({

    "true": y_true,

    "ensemble": ensemble_pred,

    "conv": pred_conv,

    "swin": pred_swin,

    "siglip": pred_sig,

    "conv_conf": conf_conv,

    "swin_conf": conf_swin,

    "sig_conf": conf_sig,

    "agreement": agree_type

})

error_df["correct"] = (
    error_df.true == error_df.ensemble
)

error_df.head()

,true,ensemble,conv,swin,siglip,conv_conf,swin_conf,sig_conf,agreement,correct
0,0,0,0,0,0,0.904741,0.951465,0.999995,3,True
1,0,0,0,0,0,0.913278,0.950820,0.999999,3,True
2,0,0,0,0,0,0.914421,0.945473,0.999996,3,True
3,0,0,0,0,0,0.949260,0.924554,0.993650,3,True
4,0,0,0,0,0,0.949947,0.948820,0.998107,3,True


In [304]:
wrong_df = error_df[
    ~error_df.correct
].copy()

print("Total Wrong :", len(wrong_df))

Total Wrong : 470


In [305]:
agreement_map = {
    3:"3 Models Agree",
    2:"2 Models Agree",
    1:"All Different"
}

summary = (
    wrong_df
    .agreement
    .map(agreement_map)
    .value_counts()
    .rename_axis("Agreement")
    .reset_index(name="Errors")
)

summary["Percent"] = (
    summary.Errors /
    len(wrong_df) *100
).round(2)

summary

,Agreement,Errors,Percent
0,2 Models Agree,277,58.94
1,3 Models Agree,179,38.09
2,All Different,14,2.98


In [306]:
cm = confusion_matrix(
    y_true,
    ensemble_pred
)

cm = pd.DataFrame(
    cm,
    index=[
        "True_Recyclable",
        "True_Electronic",
        "True_Organic"
    ],
    columns=[
        "Pred_Recyclable",
        "Pred_Electronic",
        "Pred_Organic"
    ]
)

cm

,Pred_Recyclable,Pred_Electronic,Pred_Organic
True_Recyclable,9204,54,208
True_Electronic,24,3898,8
True_Organic,166,10,11281


In [307]:
wrong_df.groupby(
    ["true","ensemble"]
).size().sort_values(
    ascending=False
)

true  ensemble
0     2           208
2     0           166
0     1            54
1     0            24
2     1            10
1     2             8
dtype: int64

In [308]:
wrong_df["conv_correct"] = (
    wrong_df.true ==
    wrong_df.conv
)

wrong_df["swin_correct"] = (
    wrong_df.true ==
    wrong_df.swin
)

wrong_df["sig_correct"] = (
    wrong_df.true ==
    wrong_df.siglip
)

pd.DataFrame({

    "Correct Cases":[

        wrong_df.conv_correct.sum(),

        wrong_df.swin_correct.sum(),

        wrong_df.sig_correct.sum()

    ]

},index=[

    "ConvNeXt",

    "Swin",

    "SigLIP"

])

,Correct Cases
ConvNeXt,108
Swin,99
SigLIP,129


In [309]:
wrong_df[[
    "conv_conf",
    "swin_conf",
    "sig_conf"
]].describe()

,conv_conf,swin_conf,sig_conf
count,470.000000,470.000000,470.000000
mean,0.803532,0.805314,0.876596
std,0.148539,0.150376,0.152524
min,0.396654,0.388512,0.380890
25%,0.703121,0.699757,0.783556
50%,0.866255,0.874471,0.952302
75%,0.925484,0.924884,0.997665
max,0.990137,0.995773,1.000000


In [310]:
wrong_df.to_csv(
    "fine_grid_errors.csv",
    index=False
)

print("Saved:", len(wrong_df))

Saved: 470


In [311]:
print("="*60)
print("TOTAL ERROR")
print("="*60)

total_error = len(wrong_df)
print("Total Wrong :", total_error)
print("Accuracy    :", 1 - total_error/len(y_true))

TOTAL ERROR
Total Wrong : 470
Accuracy    : 0.9810888021566813


In [312]:
agreement_name = {
    3:"3 Models Agree",
    2:"2 Models Agree",
    1:"All Different"
}

agreement_summary = (
    wrong_df
    .groupby("agreement")
    .size()
    .reset_index(name="Errors")
)

agreement_summary["Agreement"] = (
    agreement_summary.agreement.map(agreement_name)
)

agreement_summary["Percent"] = (
    agreement_summary.Errors /
    total_error *100
).round(2)

agreement_summary = agreement_summary[
    ["Agreement","Errors","Percent"]
].sort_values(
    "Errors",
    ascending=False
)

agreement_summary

,Agreement,Errors,Percent
1,2 Models Agree,277,58.94
2,3 Models Agree,179,38.09
0,All Different,14,2.98


In [313]:
pair_summary = (
    wrong_df
    .groupby(["true","ensemble"])
    .size()
    .reset_index(name="Count")
    .sort_values(
        "Count",
        ascending=False
    )
)

pair_summary

,true,ensemble,Count
1,0,2,208
4,2,0,166
0,0,1,54
2,1,0,24
5,2,1,10
3,1,2,8


In [314]:
labels = [
    "Recyclable",
    "Electronic",
    "Organic"
]

cm = confusion_matrix(
    y_true,
    ensemble_pred
)

cm = pd.DataFrame(
    cm,
    index=[f"True_{i}" for i in labels],
    columns=[f"Pred_{i}" for i in labels]
)

In [315]:
rescue = pd.DataFrame({

    "Model":[
        "ConvNeXt",
        "Swin",
        "SigLIP"
    ],

    "Correct_When_Ensemble_Wrong":[

        wrong_df.conv_correct.sum(),

        wrong_df.swin_correct.sum(),

        wrong_df.sig_correct.sum()

    ]

})

rescue["Rate"] = (
    rescue.Correct_When_Ensemble_Wrong /
    total_error *100
).round(2)

rescue.sort_values(
    "Correct_When_Ensemble_Wrong",
    ascending=False
)

,Model,Correct_When_Ensemble_Wrong,Rate
2,SigLIP,129,27.45
0,ConvNeXt,108,22.98
1,Swin,99,21.06


In [316]:
for model in ["conv","swin","sig"]:

    print("="*60)
    print(model.upper())

    display(

        pd.crosstab(

            wrong_df.agreement,

            wrong_df[f"{model}_correct"],

            margins=True

        )

    )

CONV


conv_correct,False,True,All
agreement,,,
1,12,2,14
2,171,106,277
3,179,0,179
All,362,108,470


SWIN


swin_correct,False,True,All
agreement,,,
1,7,7,14
2,185,92,277
3,179,0,179
All,371,99,470


SIG


sig_correct,False,True,All
agreement,,,
1,9,5,14
2,153,124,277
3,179,0,179
All,341,129,470


In [317]:
confidence_summary = wrong_df[[
    "conv_conf",
    "swin_conf",
    "sig_conf"
]].describe().T

confidence_summary

,count,mean,std,min,25%,50%,75%,max
conv_conf,470.0,0.803532,0.148539,0.396654,0.703121,0.866255,0.925484,0.990137
swin_conf,470.0,0.805314,0.150376,0.388512,0.699757,0.874471,0.924884,0.995773
sig_conf,470.0,0.876596,0.152524,0.380890,0.783556,0.952302,0.997665,1.000000


In [318]:
for model in ["conv","swin","sig"]:

    print("="*60)
    print(model.upper())

    display(

        wrong_df.groupby(
            f"{model}_correct"
        )[f"{model}_conf"].describe()

    )

CONV


,count,mean,std,min,25%,50%,75%,max
conv_correct,,,,,,,,
False,362.0,0.832668,0.130915,0.481890,0.772907,0.887280,0.930485,0.990137
True,108.0,0.705872,0.162444,0.396654,0.550969,0.682638,0.863863,0.978408


SWIN


,count,mean,std,min,25%,50%,75%,max
swin_correct,,,,,,,,
False,371.0,0.832785,0.137191,0.388512,0.772244,0.894041,0.928385,0.995773
True,99.0,0.702365,0.153608,0.439824,0.574306,0.683809,0.844745,0.947380


SIG


,count,mean,std,min,25%,50%,75%,max
sig_correct,,,,,,,,
False,341.0,0.900376,0.135967,0.380890,0.844048,0.969589,0.999007,1.000000
True,129.0,0.813734,0.175055,0.390382,0.665565,0.852104,0.986425,0.999999


In [319]:
difficulty = pd.DataFrame({

    "Total":pd.Series(y_true).value_counts().sort_index(),

    "Wrong":wrong_df.true.value_counts().sort_index()

})

difficulty.index = [
    "Recyclable",
    "Electronic",
    "Organic"
]

difficulty["Error Rate (%)"] = (
    difficulty["Wrong"] /
    difficulty["Total"] *100
).round(2)

difficulty.sort_values(
    "Error Rate (%)",
    ascending=False
)

,Total,Wrong,Error Rate (%)
Recyclable,9466,262,2.77
Organic,11457,176,1.54
Electronic,3930,32,0.81


In [320]:
wrong_df["winner"] = np.select(
    [
        wrong_df.conv_correct,
        wrong_df.swin_correct,
        wrong_df.sig_correct
    ],
    [
        "ConvNeXt",
        "Swin",
        "SigLIP"
    ],
    default="Nobody"
)

winner_summary = (
    wrong_df.winner
    .value_counts()
    .to_frame("Cases")
)

winner_summary["Percent"] = (
    winner_summary.Cases /
    total_error *100
).round(2)

winner_summary

,Cases,Percent
winner,,
Nobody,189,40.21
ConvNeXt,108,22.98
Swin,98,20.85
SigLIP,75,15.96


In [321]:
wrong_df["pattern"] = list(
    zip(
        wrong_df.conv,
        wrong_df.swin,
        wrong_df.siglip
    )
)

wrong_df["pattern"].value_counts().head(20)

pattern
(2, 2, 2)    95
(0, 0, 0)    74
(0, 2, 2)    61
(0, 2, 0)    36
(2, 2, 0)    29
(2, 0, 2)    28
(0, 0, 2)    27
(2, 0, 0)    26
(1, 1, 0)    13
(1, 0, 0)    13
(0, 1, 0)    12
(1, 1, 1)    10
(0, 1, 1)     9
(1, 0, 1)     8
(1, 0, 2)     5
(1, 2, 0)     4
(2, 1, 1)     4
(1, 2, 1)     3
(1, 1, 2)     2
(2, 0, 1)     2
Name: count, dtype: int64

In [322]:
all_df = pd.DataFrame({

    "true": y_true,

    "ensemble": ensemble_pred,

    "conv": pred_conv,

    "swin": pred_swin,

    "siglip": pred_sig

})

all_df["pattern"] = list(
    zip(
        all_df.conv,
        all_df.swin,
        all_df.siglip
    )
)

all_df.head()

,true,ensemble,conv,swin,siglip,pattern
0,0,0,0,0,0,"(0, 0, 0)"
1,0,0,0,0,0,"(0, 0, 0)"
2,0,0,0,0,0,"(0, 0, 0)"
3,0,0,0,0,0,"(0, 0, 0)"
4,0,0,0,0,0,"(0, 0, 0)"


In [323]:
pattern_stats = (
    all_df
    .groupby(["pattern","true"])
    .size()
    .unstack(fill_value=0)
)

pattern_stats

true,0,1,2
pattern,,,
"(0, 0, 0)",8594,9,65
"(0, 0, 1)",65,2,0
"(0, 0, 2)",261,0,27
"(0, 1, 0)",42,8,2
"(0, 1, 1)",11,28,0
"(0, 1, 2)",3,0,0
"(0, 2, 0)",99,0,42
"(0, 2, 1)",0,0,1
"(0, 2, 2)",81,0,125


In [324]:
pattern_stats["Total"] = pattern_stats.sum(axis=1)

pattern_stats["Majority_True"] = (
    pattern_stats[[0,1,2]]
    .idxmax(axis=1)
)

pattern_stats["Majority_Count"] = (
    pattern_stats[[0,1,2]]
    .max(axis=1)
)

pattern_stats["Purity"] = (
    pattern_stats["Majority_Count"] /
    pattern_stats["Total"]
).round(4)

pattern_stats.sort_values(
    "Total",
    ascending=False
).head(30)

true,0,1,2,Total,Majority_True,Majority_Count,Purity
pattern,,,,,,,
"(2, 2, 2)",93,2,10835,10930,2,10835,0.9913
"(0, 0, 0)",8594,9,65,8668,0,8594,0.9915
"(1, 1, 1)",8,3780,2,3790,1,3780,0.9974
"(0, 0, 2)",261,0,27,288,0,261,0.9062
"(2, 2, 0)",29,0,191,220,2,191,0.8682
"(0, 2, 2)",81,0,125,206,2,125,0.6068
"(0, 2, 0)",99,0,42,141,0,99,0.7021
"(2, 0, 2)",31,0,71,102,2,71,0.6961
"(2, 0, 0)",56,0,44,100,0,56,0.5600


In [325]:
interesting = pattern_stats[
    pattern_stats["Total"] >= 20
].sort_values(
    "Purity"
)

interesting.head(30)

true,0,1,2,Total,Majority_True,Majority_Count,Purity
pattern,,,,,,,
"(2, 0, 0)",56,0,44,100,0,56,0.5600
"(1, 0, 1)",13,17,0,30,1,17,0.5667
"(0, 2, 2)",81,0,125,206,2,125,0.6068
"(2, 0, 2)",31,0,71,102,2,71,0.6961
"(0, 2, 0)",99,0,42,141,0,99,0.7021
"(0, 1, 1)",11,28,0,39,1,28,0.7179
"(1, 1, 0)",12,49,1,62,1,49,0.7903
"(0, 1, 0)",42,8,2,52,0,42,0.8077
"(1, 0, 0)",57,8,2,67,0,57,0.8507


In [326]:
pattern_lookup = (
    pattern_stats["Majority_True"]
    .to_dict()
)

all_df["pattern_majority"] = (
    all_df["pattern"]
    .map(pattern_lookup)
)

all_df["majority_match"] = (
    all_df["ensemble"] ==
    all_df["pattern_majority"]
)

all_df["majority_match"].value_counts()

majority_match
True     24681
False      172
Name: count, dtype: int64

In [327]:
pattern_override_pred = (
    all_df["pattern"]
    .map(pattern_lookup)
    .values
)

from sklearn.metrics import accuracy_score

acc_pattern = accuracy_score(
    y_true,
    pattern_override_pred
)

print("Pattern Override Accuracy :", acc_pattern)
print("Fine Grid Accuracy        :", accuracy_score(y_true, ensemble_pred))
print("Gain :", acc_pattern - accuracy_score(y_true, ensemble_pred))

Pattern Override Accuracy : 0.9793988653281294
Fine Grid Accuracy        : 0.9810888021566813
Gain : -0.0016899368285518968


In [328]:
def apply_temperature(logits, T):
    return softmax(logits / T, axis=1)